In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
  0% 0.00/25.7M [00:00<?, ?B/s]
100% 25.7M/25.7M [00:00<00:00, 1.43GB/s]


In [ ]:
!unzip imdb-dataset-of-50k-movie-reviews.zip

Archive:  imdb-dataset-of-50k-movie-reviews.zip
  inflating: IMDB Dataset.csv        


In [ ]:
import pandas as pd
df = pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df.duplicated().sum()

np.int64(418)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(49582, 2)

In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
positive,24884
negative,24698


In [ ]:
df['sentiment'] = df['sentiment'].str.lower().map({'positive':1, 'negative':0})
df = df.dropna()
print(df['sentiment'].value_counts())

sentiment
1    24884
0    24698
Name: count, dtype: int64


Text Preprocessing

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)     # remove HTML tags
    text = re.sub(r"[^a-z0-9\s']", " ", text)  # keep letters/numbers/apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['review'].apply(clean_text)
df.head()

,review,sentiment,clean_text
0,One of the other reviewers has mentioned that ...,1,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,1,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,1,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,0,basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,petter mattei's love in the time of money is a...


building a vocabulary from the dataset and converting each text into a fixed-length sequence of token indices.
convert these indexed sequences (X) and their labels (y) into numeric form so they can be used to train a machine learning model.

In [ ]:
import numpy as np
from collections import Counter

MAX_VOCAB_SIZE = 50000
MAX_SEQ_LEN =300

def tokenize(text):
  return text.split()

counter = Counter()
for text in df['clean_text']:
  counter.update(tokenize(text))

itos = ['<PAD>','<UNK>'] + [w for w,_ in counter.most_common(MAX_VOCAB_SIZE-2)]
stoi = {w:i for i,w in enumerate(itos)}

def text_to_indices(text, max_len=MAX_SEQ_LEN):
  tokens = tokenize(text)
  ids = [stoi.get(t,1) for t in tokens]
  ids = ids[:max_len]
  ids += [0]*(max_len-len(ids))
  return ids

X = [text_to_indices(t) for t in df['clean_text']]
y = df['sentiment'].tolist()


Data Splitting into train, test and validation data

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

class IMDBDataset(Dataset):
  def __init__(self, x, y):
    self.x = x
    self.y = y

  def __len__(self):
    return len(self.x)

  def __getitem__(self, idx):
    return torch.LongTensor(self.x[idx]), torch.LongTensor([self.y[idx]])

In [ ]:
train_loader = DataLoader(IMDBDataset(x_train,y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(IMDBDataset(x_val,y_val), batch_size=64, shuffle=False)
test_loader  = DataLoader(IMDBDataset(x_test,y_test), batch_size=64, shuffle=False)

buiding models such as RNN, LSTM, and GRU for sentiment analysis.

In [ ]:
import torch.nn as nn

VOCAB_SIZE = len(itos)
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.4
NUM_CLASSES = 2

class RNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.rnn = nn.RNN(EMBED_DIM, HIDDEN_DIM, num_layers=NUM_LAYERS, batch_first=True, bidirectional=BIDIRECTIONAL)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_DIM*2, NUM_CLASSES)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, num_layers=NUM_LAYERS, batch_first=True, bidirectional=BIDIRECTIONAL)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_DIM*2, NUM_CLASSES)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

class GRUClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.gru = nn.GRU(EMBED_DIM, HIDDEN_DIM, num_layers=NUM_LAYERS, batch_first=True, bidirectional=BIDIRECTIONAL)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_DIM*2, NUM_CLASSES)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.gru(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

Doing evaluation to evaluate the model on validation and test sets and also applying earystopping based on best f1 score

In [ ]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import copy

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(DEVICE), y.to(DEVICE).view(-1)
            logits = model(x)
            pred = torch.argmax(logits, dim=1)
            y_true += y.cpu().tolist()
            y_pred += pred.cpu().tolist()
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
    return acc, prec, rec, f1


def train_model(model, train_loader, val_loader, epochs=20, patience=3):
    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    best_f1 = -1
    best_state = None
    wait = 0   # patience counter

    for epoch in range(epochs):
        model.train()
        train_losses = []
        train_y_true, train_y_pred = [], []

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE).view(-1)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            preds = torch.argmax(logits, dim=1)
            train_y_true += y.cpu().tolist()
            train_y_pred += preds.cpu().tolist()

        #Training Metrics
        train_loss = sum(train_losses) / len(train_losses)
        train_acc = accuracy_score(train_y_true, train_y_pred)

        #Validation
        val_loss_list = []
        val_y_true, val_y_pred = [], []
        model.eval()
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE).view(-1)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss_list.append(loss.item())
                pred = torch.argmax(logits, dim=1)
                val_y_true += y.cpu().tolist()
                val_y_pred += pred.cpu().tolist()

        val_loss = sum(val_loss_list) / len(val_loss_list)
        val_acc, val_prec, val_rec, val_f1 = accuracy_score(val_y_true, val_y_pred), *precision_recall_fscore_support(val_y_true, val_y_pred, average='binary')[:3]

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

        #Early Stopping (based on F1 score)
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early Stopping Triggered!")
                break

    # Load best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    return model


Training the models

In [ ]:
# Train RNN
rnn = train_model(RNNClassifier(), train_loader, val_loader)
print("RNN Test Metrics (Acc, Prec, Rec, F1):", evaluate(rnn, test_loader))

# Train LSTM
lstm = train_model(LSTMClassifier(), train_loader, val_loader)
print("LSTM Test Metrics (Acc, Prec, Rec, F1):", evaluate(lstm, test_loader))

# Train GRU
gru = train_model(GRUClassifier(), train_loader, val_loader)
print("GRU Test Metrics (Acc, Prec, Rec, F1):", evaluate(gru, test_loader))


Epoch 1/20 | Train Loss: 0.6981, Train Acc: 0.5026 | Val Loss: 0.6957, Val Acc: 0.5048, Val F1: 0.6457
Epoch 2/20 | Train Loss: 0.7002, Train Acc: 0.5051 | Val Loss: 0.6943, Val Acc: 0.5022, Val F1: 0.6545
Epoch 3/20 | Train Loss: 0.7002, Train Acc: 0.5059 | Val Loss: 0.6932, Val Acc: 0.5083, Val F1: 0.5270
Epoch 4/20 | Train Loss: 0.6978, Train Acc: 0.5108 | Val Loss: 0.6966, Val Acc: 0.5054, Val F1: 0.4875
Epoch 5/20 | Train Loss: 0.6951, Train Acc: 0.5141 | Val Loss: 0.7188, Val Acc: 0.5117, Val F1: 0.5314
Early Stopping Triggered!
RNN Test Metrics (Acc, Prec, Rec, F1): (0.5019157088122606, 0.5020478551411942, 0.9357171554841301, 0.6534792368125701)
Epoch 1/20 | Train Loss: 0.6933, Train Acc: 0.5044 | Val Loss: 0.7017, Val Acc: 0.5018, Val F1: 0.6683
Epoch 2/20 | Train Loss: 0.6928, Train Acc: 0.5100 | Val Loss: 0.6922, Val Acc: 0.5073, Val F1: 0.1733
Epoch 3/20 | Train Loss: 0.6480, Train Acc: 0.6239 | Val Loss: 0.5496, Val Acc: 0.7487, Val F1: 0.7551
Epoch 4/20 | Train Loss: 0.432

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/20 | Train Loss: 0.6953, Train Acc: 0.5064 | Val Loss: 0.6927, Val Acc: 0.4982, Val F1: 0.0000
Epoch 2/20 | Train Loss: 0.4681, Train Acc: 0.7507 | Val Loss: 0.3064, Val Acc: 0.8756, Val F1: 0.8724
Epoch 3/20 | Train Loss: 0.2211, Train Acc: 0.9127 | Val Loss: 0.2644, Val Acc: 0.8899, Val F1: 0.8893
Epoch 4/20 | Train Loss: 0.1377, Train Acc: 0.9510 | Val Loss: 0.2841, Val Acc: 0.8929, Val F1: 0.8927
Epoch 5/20 | Train Loss: 0.0728, Train Acc: 0.9777 | Val Loss: 0.3581, Val Acc: 0.8850, Val F1: 0.8816
Epoch 6/20 | Train Loss: 0.0401, Train Acc: 0.9886 | Val Loss: 0.4478, Val Acc: 0.8889, Val F1: 0.8915
Epoch 7/20 | Train Loss: 0.0279, Train Acc: 0.9923 | Val Loss: 0.4485, Val Acc: 0.8933, Val F1: 0.8938
Epoch 8/20 | Train Loss: 0.0217, Train Acc: 0.9940 | Val Loss: 0.4430, Val Acc: 0.8917, Val F1: 0.8929
Epoch 9/20 | Train Loss: 0.0168, Train Acc: 0.9957 | Val Loss: 0.5053, Val Acc: 0.8927, Val F1: 0.8928
Epoch 10/20 | Train Loss: 0.0148, Train Acc: 0.9962 | Val Loss: 0.4992, V

evluation process

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def final_test_report(model, test_loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE).view(-1)
            logits = model(x)
            pred = torch.argmax(logits, dim=1)
            y_true += y.cpu().tolist()
            y_pred += pred.cpu().tolist()

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision, Recall, F1:", precision_recall_fscore_support(y_true, y_pred, average='binary')[:3])
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, target_names=["NEGATIVE", "POSITIVE"]))
    print("\nConfusion Matrix:\n")
    print(confusion_matrix(y_true, y_pred))

In [ ]:
print("\n===== RNN TEST PERFORMANCE =====")
final_test_report(rnn, test_loader)

print("\n===== LSTM TEST PERFORMANCE =====")
final_test_report(lstm, test_loader)

print("\n===== GRU TEST PERFORMANCE =====")
final_test_report(gru, test_loader)


===== RNN TEST PERFORMANCE =====
Accuracy: 0.5019157088122606
Precision, Recall, F1: (0.5020478551411942, 0.9357171554841301, 0.6534792368125701)

Classification Report:

              precision    recall  f1-score   support

    NEGATIVE       0.50      0.06      0.11      2470
    POSITIVE       0.50      0.94      0.65      2489

    accuracy                           0.50      4959
   macro avg       0.50      0.50      0.38      4959
weighted avg       0.50      0.50      0.39      4959


Confusion Matrix:

[[ 160 2310]
 [ 160 2329]]

===== LSTM TEST PERFORMANCE =====
Accuracy: 0.8786045573704376
Precision, Recall, F1: (0.884004884004884, 0.8726396143029329, 0.8782854832187627)

Classification Report:

              precision    recall  f1-score   support

    NEGATIVE       0.87      0.88      0.88      2470
    POSITIVE       0.88      0.87      0.88      2489

    accuracy                           0.88      4959
   macro avg       0.88      0.88      0.88      4959
weighted a

In [ ]:
def predict(model, text):
    model.eval()
    text = clean_text(text)
    indices = text_to_indices(text)
    x = torch.tensor([indices]).to(DEVICE)

    with torch.no_grad():
        logits = model(x)
        pred = torch.argmax(logits, dim=1).item()

    return "POSITIVE" if pred == 1 else "NEGATIVE"

In [ ]:
print("RNN Prediction:")
print(predict(rnn, "I loved this movie, fantastic!"))
print(predict(rnn, "It is horrible movie. i hate this movie"))

print("\nLSTM Prediction:")
print(predict(lstm, "I loved this movie, fantastic!"))
print(predict(lstm, "Worst film ever made."))

print("\nGRU Prediction:")
print(predict(gru, "I loved this movie, fantastic!"))
print(predict(gru, "Worst film ever made."))

RNN Prediction:
POSITIVE
POSITIVE

LSTM Prediction:
POSITIVE
NEGATIVE

GRU Prediction:
POSITIVE
NEGATIVE


As we can see here , RNN is not able to classify perfectly compare to other 2 models. Whereas other two models are predicting correctly.